In [1]:
import re

with open("data/medicina.xml",encoding="utf-8") as f: 
    texto = f.read()
texto = re.sub(r"<b>\s*(\d+) ", r"<b>@@\1 ", texto)

conceito = re.split(r"<b>\s*@", texto)


In [2]:
def processar_conceito(c):

    # extract concept id
    id = re.search(r"^@(\d+)", c)

    # normalize markers
    c = re.sub(r"SIN\.-", r"@SIN.-", c)
    c = re.sub(r"VAR\.-", r"@VAR.-", c)
    c = re.sub(r"Nota\.-", r"@Nota.-", c)

    # extract optional fields
    sin  = re.search(r"@SIN\.-([^@<]+)", c)
    var  = re.search(r"@VAR\.-([^@<]+)", c)
    nota = re.search(r"@Nota\.-([^@<]+)", c)

    # extract Galician term
    trGal = re.search(r"^@\d+([\w ]+)</b>", c)

    #domain
    dom = re.search(r'font="6"><i>(.*?)</i>', c)

    # language
    blocos = re.findall(
        r'>\s*(en|pt|es|la)\s*</text>\n(.*?)(?=>\s*(?:en|pt|es|la)\s*</text>|<b>)',
        c,
        re.DOTALL
    )

    ling = []
    for lang, bloco in blocos:
        termini = re.findall(r'<i>([^<]+)', bloco)
        t = " ".join(i.strip() for i in termini)
        ling.append((lang, t))

    res = {}
    # if no id found, skip concept
    if not id:
        return {}, None

    if nota:
        res["nota"] = nota.group(1)

    if var:
        res["var"] = var.group(1)

    if sin:
        res["sin"] = sin.group(1)

    if trGal:
        res["galego"] = trGal.group(1)

    if dom:
        res["dom"] = dom.group(1)

    for l, t in ling:
        res[l] = t

    return res, id.group(1)





In [3]:
entries = {}
for c in conceito[1:]:
    res, id = processar_conceito(c)
    entries[id] = res


In [4]:
import json
f_out = open("dicionario_medicina.json", "w")
json.dump(entries, f_out, indent=4, ensure_ascii=False)